# MuseTalk — Audio-Driven Lip Sync
**InterVyu.ai | Avatar-Video**

Based on [TMElyralab/MuseTalk](https://github.com/TMElyralab/MuseTalk).

**Environments supported:**
- Google Colab (Tesla T4/V100, Python 3.12)
- Local macOS/Linux with  (Python 3.11)


## 0. Repo Setup (Colab only)
Skip this cell if running locally.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Option A: clone from GitHub (replace with your repo URL)
    !git clone https://github.com/YOUR_ORG/Avatar-Video.git
    %cd Avatar-Video

    # Option B: mount Google Drive
    # from google.colab import drive
    # drive.mount("/content/drive")
    # %cd /content/drive/MyDrive/InterVyu.ai/Avatar-Video

print(f"Working directory: {os.getcwd()}")
print(f"Running in Colab: {IN_COLAB}")

MessageError: User cancelled dfs_ephemeral authorization

## 1. Install Dependencies

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # --- Colab (Python 3.12, pip) ---
    # Core deps — use unpinned versions compatible with Python 3.12
    !pip install -q \n
        diffusers==0.30.2 \n
        accelerate==0.28.0 \n
        transformers==4.39.2 \n
        "huggingface_hub>=0.30.2" \n
        opencv-python \n
        soundfile \n
        librosa \n
        einops \n
        gradio \n
        omegaconf \n
        ffmpeg-python \n
        moviepy \n
        "imageio[ffmpeg]" \n
        gdown

    # openmim without its transitive junk (openxlab downgrades setuptools)
    !pip install -q openmim --no-deps
    !pip install -q -U setuptools  # restore setuptools after openmim

    # MMLab packages via mim
    !mim install -q mmengine
    !mim install -q "mmcv==2.0.1"
    !mim install -q "mmdet==3.1.0"
    !mim install -q "mmpose==1.1.0"

else:
    # --- Local (Python 3.11, uv) ---
    !uv sync --python /opt/homebrew/Caskroom/miniconda/base/bin/python3.11
    # MMLab must still go through mim
    !uv run pip install -q openmim --no-deps
    !uv run pip install -q -U setuptools
    !uv run mim install -q mmengine
    !uv run mim install -q "mmcv==2.0.1"
    !uv run mim install -q "mmdet==3.1.0"
    !uv run mim install -q "mmpose==1.1.0"

print("Install complete.")

## 2. Download Model Weights (~4 GB)

In [ ]:
!bash download_models.sh

## 3. Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("Apple MPS (Metal) — will work, slower than CUDA")
else:
    print("CPU only — very slow, GPU strongly recommended")

## 4. Load Pipeline

In [ ]:
from musetalk_pipeline import MuseTalkPipeline, MuseTalkConfig

VERSION = "v1"  # "v1" or "v15" (v15 = higher quality, needs more VRAM)

cfg = MuseTalkConfig(version=VERSION)
if VERSION == "v15":
    cfg.unet_model_path  = "models/musetalkV15/unet.pth"
    cfg.unet_config_path = "models/musetalkV15/musetalk.json"

pipe = MuseTalkPipeline(cfg)
pipe.load_models()
print("Pipeline ready!")

## 5. Run Inference

In [ ]:
VIDEO_PATH  = "data/video/source.mp4"
AUDIO_PATH  = "data/audio/target.wav"
OUTPUT_PATH = "results/output.mp4"
BBOX_SHIFT  = 0  # tune if lips look misaligned (-10 to 10)

result = pipe.run(
    video_path=VIDEO_PATH,
    audio_path=AUDIO_PATH,
    output_path=OUTPUT_PATH,
    bbox_shift=BBOX_SHIFT,
)
print(f"Saved to: {result}")

## 6. Preview

In [ ]:
from IPython.display import Video
Video(OUTPUT_PATH, embed=True, width=640)

## 7. Batch Inference via YAML Config

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
run_prefix = "" if IN_COLAB else "uv run "
!{run_prefix}python -m scripts.inference \n
    --inference_config configs/inference/test.yaml \n
    --result_dir results/batch \n
    --version v1

## 8. Gradio Web UI

In Colab, enable  for a public tunnel link.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Patch app.py to use share=True for Colab tunnel
    !sed -i "s/demo.launch(share=False)/demo.launch(share=True)/" app.py
run_prefix = "" if IN_COLAB else "uv run "
!{run_prefix}python app.py

## Tips

| Issue | Fix |
|---|---|
| Lips misaligned | Adjust  (-10 to +10) |
| Low resolution | Post-process with GFPGAN super-resolution |
| Flickering | Use  |
| GPU OOM | Reduce  in  |
| Colab disconnects | Save models to Google Drive and reload |
| Slow on MPS | Apple M-series: expect ~6–10 fps vs 30 fps on CUDA |
